# Demo 01 : A research agent

We use local manuals and service tickets to answer a question that spans both sources. Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents==0.7.24", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · research-agent

We keep the corpus small and search only local files. The graph separates planning, parallel research and report writing, following the shape of `open_deep_research` without web access.

In [2]:
import csv
from pathlib import Path

from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from typing_extensions import TypedDict
import operator
from typing import Annotated
from pydantic import BaseModel

manuals_dir = Path(data_folder("demo-world/manuals"))
manuals = [(p.name, p.read_text(encoding="utf-8")) for p in sorted(manuals_dir.glob("*.md"))]
with open(data("demo-world/tickets.csv"), encoding="utf-8") as f:
    tickets = list(csv.DictReader(f))

def search(query, limit=2):
    words = set(query.lower().split())
    ranked = sorted(manuals, key=lambda row: sum(row[1].lower().count(w) for w in words), reverse=True)
    return [(name, body) for name, body in ranked[:limit]]

QUESTION = "Which repeated fault took the most recorded mechanic hours, and what local procedure applies?"
hours = {}
for row in tickets:
    key = (row["error_code"], row["symptom"])
    hours[key] = hours.get(key, 0) + float(row["hours"])
ticket_facts = sorted(hours.items(), key=lambda item: item[1], reverse=True)[:4]
print("manual files:", len(manuals), "ticket rows:", len(tickets))
print("top ticket groups:", ticket_facts[:2])

manual files: 31 ticket rows: 43
top ticket groups: [(('F-RR-120', 'Door controller not responding reported by drivers'), 27.5), (('F-MM-210', 'Ramp stops on battery power reported by drivers'), 16.0)]


A planner divides the question into a ticket query and a manual query. We run fixed local retrieval for each branch, then ask the model to interpret only the retrieved facts.

In [3]:
class Plan(BaseModel):
    questions: list[str]

plan = llm.with_structured_output(Plan).invoke(
    "Return JSON with two short questions, one for tickets and one for manuals. "
    "The goal is to answer: " + QUESTION)
questions = plan.questions
if len(questions) < 2:
    questions = ["Which fault has the most hours in the tickets?", "Which procedure covers the leading fault?"]
branches = [
    {"source": "tickets", "question": questions[0], "context": str(ticket_facts), "files": ["tickets.csv"]},
    {"source": "manuals", "question": questions[1],
     "context": "\n\n".join(f"[{n}]\n{body[:1000]}" for n, body in search("F-CE-410 traction over-temperature procedure")),
     "files": [n for n, _ in search("F-CE-410 traction over-temperature procedure")]},
]
print("research branches:", [(b["source"], b["question"]) for b in branches])

research branches: [('tickets', 'Which repeated fault took the most recorded mechanic hours in the tickets?'), ('manuals', 'What local procedure applies to that fault, according to the manuals?')]


In [4]:
class ResearchState(TypedDict):
    question: str
    branches: list[dict]
    notes: Annotated[list[dict], operator.add]
    report: str

def fan_out(state):
    return [Send("research", branch) for branch in state["branches"]]

def research(branch):
    answer = llm.invoke([
        SystemMessage("Use only the supplied context. State what it supports and name the source file."),
        HumanMessage(f"Question: {branch['question']}\nFiles: {branch['files']}\nContext: {branch['context']}")
    ]).content
    return {"notes": [{"source": branch["source"], "files": branch["files"], "finding": answer}]}

def write_report(state):
    notes = "\n".join(f"{n['finding']} [" + ", ".join(n["files"]) + "]" for n in state["notes"])
    report = llm.invoke("Answer the question using only these notes. Keep file citations beside claims. "
                        f"Question: {state['question']}\nNotes:\n{notes}").content
    return {"report": report}

graph = StateGraph(ResearchState)
graph.add_node("research", research)
graph.add_node("write_report", write_report)
graph.add_conditional_edges(START, fan_out, ["research"])
graph.add_edge("research", "write_report")
graph.add_edge("write_report", END)
result = graph.compile().invoke({"question": QUESTION, "branches": branches, "notes": [], "report": ""})
print("report:\n", result["report"])
print("sources:", sorted({name for note in result["notes"] for name in note["files"]}))

report:
 The repeated fault with the most recorded mechanic time was **F-RR-120 — Door controller not responding**, at **27.5 hours**. [tickets.csv]

The notes don’t identify a local procedure for F-RR-120. They describe procedures for different faults: **PROC-CE-CF-02** for traction cooling fan replacement and **PROC-CE-TB-01** for traction battery module replacement. [ce-12-procedure-ce-cf-02.md, ce-12-procedure-ce-tb-01.md]
sources: ['ce-12-procedure-ce-cf-02.md', 'ce-12-procedure-ce-tb-01.md', 'tickets.csv']


A source list checks provenance, not whether each source supports the sentence beside it. We read the cited passage before using the report as an operational recommendation.

## Things to try

- Ask about a fault code found in the ticket file
- Change retrieval to return one manual and compare the evidence